In [ ]:
import os
import re
import time
import csv
import subprocess
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

FACEBOOK_EMAIL = "pkongla478@gmail.com" 
FACEBOOK_PASSWORD = "140846Zx@d"
OUTPUT_CSV_FILE = 'facebook_group_posts.csv'

GROUP_URLS = [
    "https://www.facebook.com/groups/302468990428489/",
    "https://www.facebook.com/groups/322977734828852/",
    "https://www.facebook.com/groups/812156038944325/",
    "https://www.facebook.com/groups/1472146056424210/",
    "https://www.facebook.com/groups/426467944402414/",
    "https://www.facebook.com/groups/homerentcm/",
    "https://www.facebook.com/groups/509895225859790/",
    "https://www.facebook.com/groups/1299302030158649/",
    "https://www.facebook.com/groups/530492663958652/",
    "https://www.facebook.com/groups/596670275854317/",
    "https://www.facebook.com/groups/cnxre/",
    "https://www.facebook.com/groups/1885263611797363/",
    "https://www.facebook.com/groups/1897854047114064/",
    "https://www.facebook.com/groups/303531690102574/",
    "https://www.facebook.com/groups/1881982752029163/",
    "https://www.facebook.com/groups/568026117396809/",
    "https://www.facebook.com/groups/1928645537294336/",
    "https://www.facebook.com/groups/298821664628156/",
    "https://www.facebook.com/groups/903971886395138/",
    "https://www.facebook.com/groups/142702946428033/members",
    "https://www.facebook.com/groups/250469775470436/",
    "https://www.facebook.com/groups/1873094122912006/",
    "https://www.facebook.com/groups/2083392538672247/",
    "https://www.facebook.com/groups/864193946960728/",
    "https://www.facebook.com/groups/baanchiangmai/",
    "https://www.facebook.com/groups/694087674785430/",
    "https://www.facebook.com/groups/411301775702951/",
    "https://www.facebook.com/groups/169718747164928/",
    "https://www.facebook.com/groups/1475061816108017/",
    "https://www.facebook.com/groups/1582425938465943/",
    "https://www.facebook.com/groups/sale.rent.poolvillachiangmai/",
    "https://www.facebook.com/groups/251125079442673/",
    "https://www.facebook.com/groups/1034329704984830/",
    "https://www.facebook.com/groups/203683550205761/",
    "https://www.facebook.com/groups/1450131905304596/",
    "https://www.facebook.com/groups/959493160788393/",
    "https://www.facebook.com/groups/2897034136980512/",
    "https://www.facebook.com/groups/korn.property/",
    "https://www.facebook.com/groups/1456428424593312/",
    "https://www.facebook.com/groups/152080739566471/",
    "https://www.facebook.com/groups/Land.House.C.M.2014/",
    "https://www.facebook.com/groups/2336780789695894/",
    "https://www.facebook.com/groups/236116797208244/",
    "https://www.facebook.com/groups/landhomechiangmai/"
]

def install_chrome_environment():
    chrome_path = os.path.join(os.getcwd(), "chrome-linux64", "chrome")
    driver_path = os.path.join(os.getcwd(), "chromedriver-linux64", "chromedriver")
    
    if not os.path.exists(chrome_path) or not os.path.exists(driver_path):
        subprocess.run(["apt-get", "update"], check=True)
        # Download Chrome for Testing (Browser)
        subprocess.run(["wget", "-N", "https://storage.googleapis.com/chrome-for-testing-public/133.0.6943.53/linux64/chrome-linux64.zip"], check=True)
        subprocess.run(["unzip", "-o", "chrome-linux64.zip"], check=True)
        
        # Download Chrome for Testing (Driver)
        subprocess.run(["wget", "-N", "https://storage.googleapis.com/chrome-for-testing-public/133.0.6943.53/linux64/chromedriver-linux64.zip"], check=True)
        subprocess.run(["unzip", "-o", "chromedriver-linux64.zip"], check=True)
        
        subprocess.run(["chmod", "+x", chrome_path], check=True)
        subprocess.run(["chmod", "+x", driver_path], check=True)
    
    return chrome_path, driver_path

def login_to_facebook(driver):
    driver.get("https://www.facebook.com")
    time.sleep(3)
    selectors = [
        "button[data-cookiebanner='accept_button_dialog']",
        "button[title='Allow all cookies']",
        "button[title='Accept All']",
        "button[aria-label='Allow all cookies']"
    ]
    for s in selectors:
        btns = driver.find_elements(By.CSS_SELECTOR, s)
        if btns and btns[0].is_displayed():
            btns[0].click()
            time.sleep(2)
            break
    
    email = driver.find_elements(By.ID, "email")
    pwd = driver.find_elements(By.ID, "pass")
    
    if email and pwd:
        email[0].clear()
        email[0].send_keys(FACEBOOK_EMAIL)
        pwd[0].send_keys(FACEBOOK_PASSWORD)
        pwd[0].send_keys(Keys.RETURN)
        WebDriverWait(driver, 30).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div[role='banner'], div[aria-label='Facebook']"))
        )

def parse_relative_time_to_days(time_str):
    s = (time_str or "").strip().lower()
    if not s:
        return 999
    if any(x in s for x in ["เมื่อสักครู่", "just now", "นาที", "minute", "min", "ชม", "ชั่วโมง", "hour", "hr"]):
        return 0
    m = re.search(r'(\d+)', s)
    n = int(m.group(1)) if m else 0
    if "วัน" in s or "day" in s or "d" == s[-1:]:
        return n
    if "สัปดาห์" in s or "week" in s or "wk" in s or "w" == s[-1:]:
        return n * 7
    if "เดือน" in s or "month" in s or "mo" in s:
        return n * 30
    if "ปี" in s or "year" in s or "yr" in s:
        return n * 365
    return 999

def apply_new_post_filter(driver):
    triggers = driver.find_elements(By.XPATH, "//span[contains(text(),'กิจกรรมล่าสุด') or contains(text(),'Recent activity') or contains(text(),'New activity') or contains(text(),'รายการสินค้าใหม่')]")
    if triggers:
        driver.execute_script("arguments[0].click();", triggers[0])
        time.sleep(2)
        new_posts = driver.find_elements(By.XPATH, "//div[@role='menuitemradio']//span[contains(text(),'โพสต์ใหม่') or contains(text(),'New posts') or contains(text(),'แสดงโพสต์ล่าสุดก่อน') or contains(text(),'แสดงรายการสินค้าล่าสุดก่อน')]")
        if new_posts:
            driver.execute_script("arguments[0].click();", new_posts[0])
            time.sleep(5)
            driver.execute_script("window.scrollTo(0, 0);")
            time.sleep(2)

def normalize_group_url(u):
    v = u.strip()
    v = v.replace("m.facebook.com", "www.facebook.com")
    if "www.facebook.com" not in v:
        v = v.replace("facebook.com", "www.facebook.com")
    if "/members" in v:
        v = v.split("/members")[0] + "/"
    if not v.endswith("/"):
        v = v + "/"
    return v

def collect_group_post_urls(driver, group_url):
    group_url = normalize_group_url(group_url)
    driver.get(group_url)
    time.sleep(4)
    apply_new_post_filter(driver)
    
    seen = set()
    results = []
    prev_len = 0
    stagnant = 0
    loops = 0
    stop = False
    
    while True:
        data = driver.execute_script("""
            function absUrl(href){ if(!href) return null; if(href.indexOf('http')===0) return href.split('?')[0]; return location.origin + href.split('?')[0]; }
            var posts = Array.from(document.querySelectorAll('div[role="feed"] > div, div[role="article"], div.x1yztbdb'));
            var out = [];
            for (var i=0;i<posts.length;i++){
                var p = posts[i];
                var a = p.querySelector("a[href*='/posts/'], a[href*='/permalink/']");
                if(!a) continue;
                var href = absUrl(a.getAttribute('href'));
                if(!href) continue;
                var t = p.querySelector("a[aria-label], span[id*='jsc_c'], span.x193iq5w");
                var tlabel = "";
                if(t) tlabel = t.textContent; 
                if (!tlabel) {
                     var spans = p.querySelectorAll('span');
                     for(var j=0; j<spans.length; j++){
                        if(spans[j].textContent.match(/(hr|min|day|นาที|ชั่วโมง|วัน)/)) {
                            tlabel = spans[j].textContent;
                            break;
                        }
                     }
                }
                out.push([href, tlabel]);
            }
            return out;
        """)
        
        if data:
            for href, tlabel in data:
                if href in seen:
                    continue
                if any(x in href for x in ["/reel/", "/videos/", "/watch/", "video_id"]):
                    continue
                if "/posts/" not in href and "/permalink/" not in href:
                    continue
                days_ago = parse_relative_time_to_days(tlabel)
                seen.add(href)
                if days_ago == 0:
                    results.append(href)
                elif days_ago > 0 and days_ago != 999:
                    stop = True
        
        if stop:
            break
            
        driver.execute_script("window.scrollBy(0, 1000);")
        time.sleep(1.5)
        
        curr_len = len(seen)
        if curr_len == prev_len:
            stagnant += 1
        else:
            stagnant = 0
        prev_len = curr_len
        loops += 1
        
        if stagnant >= 5 or loops >= 100:
            break
            
    return results

def main():
    chrome_bin, driver_bin = install_chrome_environment()

    options = uc.ChromeOptions()
    options.add_argument("--disable-notifications")
    options.add_argument("--lang=en-US")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--no-sandbox")
    options.add_argument("--headless=new")
    options.add_argument("--window-size=1920,1080")
    options.binary_location = chrome_bin
    
    driver = uc.Chrome(
        options=options, 
        driver_executable_path=driver_bin,
        version_main=133
    )
    driver.set_page_load_timeout(90)
    driver.set_script_timeout(90)
    
    login_to_facebook(driver)
    
    headers = ['ชื่อกลุ่ม', 'ลิงค์', 'PostURL']
    file_exists = os.path.isfile(OUTPUT_CSV_FILE)
    
    with open(OUTPUT_CSV_FILE, 'a', newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(f, fieldnames=headers)
        if not file_exists:
            writer.writeheader()
            
    for i, link in enumerate(GROUP_URLS, start=1):
        urls = collect_group_post_urls(driver, link)
        new_rows = []
        group_name = link.split("/groups/")[1].split("/")[0]
        
        for u in urls:
            new_rows.append({'ชื่อกลุ่ม': group_name, 'ลิงค์': link, 'PostURL': u})
            
        if new_rows:
            with open(OUTPUT_CSV_FILE, 'a', newline='', encoding='utf-8-sig') as f:
                writer = csv.DictWriter(f, fieldnames=headers)
                writer.writerows(new_rows)
        
        time.sleep(1.0)
        
    driver.quit()

if __name__ == "__main__":
    main()